# AgriN — OOD Centroid Computation

**Purpose:** the disease model (EfficientNet-B3, 38 PlantVillage classes) always outputs a confident prediction via softmax, even for images completely outside its training distribution — a different plant species, a disease it's never seen, a real-world outdoor photo. Softmax has no "none of the above" option; it is mathematically forced to sum to 100% across the 38 known classes no matter what.

This notebook computes, once, offline, from the already-trained model:
- A **centroid** (average feature vector) for each of the 38 classes, using the 1536-dim embedding right before the classifier head — a richer signal than the final softmax output.
- A **distance threshold**, from how far real, correctly-classified training images typically sit from their own class's centroid.

At runtime (in `main.py`), a new image's feature vector is compared to its nearest centroid. If the distance is far beyond what real training images ever showed, that's a strong, cheap, independent signal that the image is likely out-of-distribution — regardless of how confident the softmax number looks.

**This is inference only — no training.** Uses your already-trained `best_model.pth`.

**Setup:** Attach dataset `abdallahalidev/plantvillage-dataset` (for images) and your `crop_disease` dataset (for `best_model.pth` + `label_map.json`). GPU accelerator recommended but not required — this is fast even on CPU since it's inference-only over a modest sample.

In [33]:
import os
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import efficientnet_b3
from PIL import Image

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

PyTorch: 2.10.0+cu128
CUDA available: True
Using device: cuda


## 1. Config — matches your attached dataset paths

In [34]:
import os
for root in os.listdir("/kaggle/input/datasets"):
    print(root)
    sub = os.path.join("/kaggle/input/datasets", root)
    if os.path.isdir(sub):
        for item in os.listdir(sub):
            print("  ", item)

dot999
   crop-disease
abdallahalidev
   plantvillage-dataset


In [35]:
DATA_DIR = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
MODEL_PATH = "/kaggle/input/datasets/dot999/crop-disease/best_model.pth"
LABEL_MAP_PATH = "/kaggle/input/datasets/dot999/crop-disease/label_map.json"

SAMPLES_PER_CLASS = 60   # enough for a stable centroid; raise if you want tighter estimates
SEED = 42

OUTPUT_PATH = "/kaggle/working/ood_centroids.json"

random.seed(SEED)

assert os.path.exists(MODEL_PATH), f"Model not found at {MODEL_PATH} — check the crop_disease dataset is attached"
assert os.path.exists(LABEL_MAP_PATH), f"Label map not found at {LABEL_MAP_PATH}"
assert os.path.isdir(DATA_DIR), f"PlantVillage color folder not found at {DATA_DIR} — check dataset is attached"
print("All paths verified.")

All paths verified.


## 2. Rebuild the exact trained architecture and load weights

This matches the architecture in `main.py` exactly — EfficientNet-B3 with the custom BatchNorm+Linear+ReLU+Dropout+Linear head. No training happens here, only `load_state_dict` + `eval()`.

In [36]:
with open(LABEL_MAP_PATH) as f:
    LABEL_MAP = json.load(f)

NUM_CLASSES = len(LABEL_MAP)
IDX_TO_CLASS_NAME = {int(k): v["class_name"] for k, v in LABEL_MAP.items()}
print(f"Loaded label map: {NUM_CLASSES} classes")

model = efficientnet_b3(weights=None)
model.classifier = nn.Sequential(
    nn.BatchNorm1d(1536, eps=0.001, momentum=0.01),
    nn.Linear(1536, 256),
    nn.ReLU(),
    nn.Dropout(p=0.45),
    nn.Linear(256, NUM_CLASSES),
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE).eval()
print("Model loaded and set to eval mode.")

Loaded label map: 38 classes
Model loaded and set to eval mode.


## 3. Attach a forward hook to capture the 1536-dim embedding

`classifier[0]` is the `BatchNorm1d(1536)` — its output is the pooled, normalized feature vector fed into the rest of the classifier head. Hooking here captures embeddings without modifying the model at all.

In [37]:
captured = {}

def hook(module, inp, output):
    captured["features"] = output.detach()

hook_handle = model.classifier[0].register_forward_hook(hook)
print("Feature-extraction hook attached to classifier[0] (BatchNorm1d).")

Feature-extraction hook attached to classifier[0] (BatchNorm1d).


In [38]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 4. Run a sample of each class through the model, collect embeddings

In [39]:
class_features = {}
start = time.time()

for idx in range(NUM_CLASSES):
    class_name = IDX_TO_CLASS_NAME[idx]
    class_dir = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_dir):
        print(f"  \u26a0 WARNING: folder not found for class '{class_name}' — skipping")
        continue

    files = [f for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not files:
        print(f"  \u26a0 WARNING: no images found for class '{class_name}' — skipping")
        continue

    random.shuffle(files)
    sample_files = files[:SAMPLES_PER_CLASS]

    feats = []
    with torch.no_grad():
        for fname in sample_files:
            try:
                img = Image.open(os.path.join(class_dir, fname)).convert("RGB")
                inp = transform(img).unsqueeze(0).to(DEVICE)
                _ = model(inp)  # triggers the hook
                feats.append(captured["features"].cpu().numpy().flatten())
            except Exception as e:
                print(f"    (skipped unreadable file {fname}: {e})")

    if feats:
        class_features[class_name] = feats
        print(f"  [{idx+1:2d}/{NUM_CLASSES}] {class_name:45s} — {len(feats)} images embedded")

hook_handle.remove()
elapsed = time.time() - start
print(f"\nFeature extraction done in {elapsed/60:.1f} min for {len(class_features)} classes")

  [ 1/38] Apple___Apple_scab                            — 60 images embedded
  [ 2/38] Apple___Black_rot                             — 60 images embedded
  [ 3/38] Apple___Cedar_apple_rust                      — 60 images embedded
  [ 4/38] Apple___healthy                               — 60 images embedded
  [ 5/38] Blueberry___healthy                           — 60 images embedded
  [ 6/38] Cherry_(including_sour)___Powdery_mildew      — 60 images embedded
  [ 7/38] Cherry_(including_sour)___healthy             — 60 images embedded
  [ 8/38] Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot — 60 images embedded
  [ 9/38] Corn_(maize)___Common_rust_                   — 60 images embedded
  [10/38] Corn_(maize)___Northern_Leaf_Blight           — 60 images embedded
  [11/38] Corn_(maize)___healthy                        — 60 images embedded
  [12/38] Grape___Black_rot                             — 60 images embedded
  [13/38] Grape___Esca_(Black_Measles)                  — 60 images emb

## 5. Compute per-class centroids and the global OOD distance threshold

In [40]:
centroids = {}
all_intra_distances = []

for class_name, feats in class_features.items():
    feats_arr = np.stack(feats)  # (n_samples, 1536)
    centroid = feats_arr.mean(axis=0)
    distances = np.linalg.norm(feats_arr - centroid, axis=1)

    centroids[class_name] = {
        "centroid": centroid.tolist(),
        "mean_intra_distance": float(distances.mean()),
        "p95_intra_distance": float(np.percentile(distances, 95)),
        "n_samples": len(feats),
    }
    all_intra_distances.extend(distances.tolist())

# Global threshold: a new image is flagged OOD if its distance to the
# NEAREST class centroid exceeds this. The 95th percentile of all observed
# in-distribution intra-class distances is a reasonable, slightly
# conservative cutoff — real training images occasionally sit this far
# from their own centroid, so anything meaningfully beyond it is a
# genuinely unusual signal, not normal variation.
global_threshold = float(np.percentile(all_intra_distances, 96))

print(f"Computed centroids for {len(centroids)}/{NUM_CLASSES} classes")
print(f"Global OOD distance threshold: {global_threshold:.2f}")
print(f"(mean intra-class distance across all classes: {np.mean(all_intra_distances):.2f})")

Computed centroids for 38/38 classes
Global OOD distance threshold: 28.58
(mean intra-class distance across all classes: 20.16)


## 6. Sanity check — verify a few known images score well, isn't required but reassuring

In [41]:
# Quick sanity check: pick one held-out image per class (not necessarily one
# used in the centroid sample) and confirm its nearest-centroid distance is
# comfortably within the threshold — i.e. the threshold isn't so tight that
# it would falsely flag genuine in-distribution images.
import random as _r

centroid_matrix = np.stack([np.array(c["centroid"]) for c in centroids.values()])
centroid_names = list(centroids.keys())

sample_classes = _r.sample(list(class_features.keys()), min(5, len(class_features)))
false_flag_count = 0

with torch.no_grad():
    for class_name in sample_classes:
        class_dir = os.path.join(DATA_DIR, class_name)
        files = [f for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        test_file = _r.choice(files)
        img = Image.open(os.path.join(class_dir, test_file)).convert("RGB")
        inp = transform(img).unsqueeze(0).to(DEVICE)
        hook_handle = model.classifier[0].register_forward_hook(hook)
        _ = model(inp)
        hook_handle.remove()
        feat = captured["features"].cpu().numpy().flatten()

        dists = np.linalg.norm(centroid_matrix - feat, axis=1)
        nearest_idx = np.argmin(dists)
        nearest_class = centroid_names[nearest_idx]
        nearest_dist = dists[nearest_idx]
        is_flagged = nearest_dist > global_threshold
        if is_flagged:
            false_flag_count += 1

        match = "\u2713" if nearest_class == class_name else "\u2717 (different nearest class!)"
        flag_str = "FLAGGED AS OOD (unexpected!)" if is_flagged else "in-distribution, OK"
        print(f"{class_name[:40]:40s} -> nearest: {nearest_class[:40]:40s} {match} | dist={nearest_dist:.2f} | {flag_str}")

print(f"\n{false_flag_count}/{len(sample_classes)} known-good images were incorrectly flagged as OOD.")
print("If this is 0, the threshold looks reasonable. If several are flagged, consider raising the")
print("threshold percentile (e.g. 97th or 99th instead of 95th) before saving.")

Apple___Apple_scab                       -> nearest: Apple___Apple_scab                       ✓ | dist=22.49 | in-distribution, OK
Grape___Leaf_blight_(Isariopsis_Leaf_Spo -> nearest: Grape___Leaf_blight_(Isariopsis_Leaf_Spo ✓ | dist=16.86 | in-distribution, OK
Corn_(maize)___Cercospora_leaf_spot Gray -> nearest: Corn_(maize)___Cercospora_leaf_spot Gray ✓ | dist=19.36 | in-distribution, OK
Apple___Cedar_apple_rust                 -> nearest: Apple___Cedar_apple_rust                 ✓ | dist=28.08 | in-distribution, OK
Tomato___Tomato_Yellow_Leaf_Curl_Virus   -> nearest: Tomato___Tomato_Yellow_Leaf_Curl_Virus   ✓ | dist=18.24 | in-distribution, OK

0/5 known-good images were incorrectly flagged as OOD.
If this is 0, the threshold looks reasonable. If several are flagged, consider raising the
threshold percentile (e.g. 97th or 99th instead of 95th) before saving.


## 7. Save the centroids file

In [42]:
output_data = {
    "feature_dim": 1536,
    "global_ood_threshold": global_threshold,
    "note": (
        "At inference, compute the new image's feature vector, find its "
        "nearest centroid (Euclidean distance) among all classes, and "
        "compare to global_ood_threshold. Beyond it -> flag as likely "
        "out-of-distribution (not one of the 38 trained classes), "
        "regardless of the softmax confidence reported."
    ),
    "class_centroids": centroids,
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(output_data, f)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved to {OUTPUT_PATH}")
print(f"File size: {size_mb:.2f} MB")
print(f"\nDownload ood_centroids.json from the Kaggle 'Output' panel (/kaggle/working/)")
print(f"and place it at models/ood_centroids.json in your backend repo.")

Saved to /kaggle/working/ood_centroids.json
File size: 1.22 MB

Download ood_centroids.json from the Kaggle 'Output' panel (/kaggle/working/)
and place it at models/ood_centroids.json in your backend repo.
